# CySent Unsloth Training (Colab)A minimal notebook to fine-tune a small model for CySent action prediction.

In [ ]:
!pip -q install unsloth transformers datasets trl peft accelerate bitsandbytes# Optional: clone project# !git clone https://github.com/<your-org>/CySent.git# %cd CySent

In [ ]:
from pathlib import Pathfrom datasets import load_datasetfrom unsloth import FastLanguageModelfrom trl import SFTTrainerfrom transformers import TrainingArgumentsdataset_path = Path('datasets/cysent_action_dataset.jsonl')ds = load_dataset('json', data_files=str(dataset_path), split='train')split = ds.train_test_split(test_size=0.05, seed=42)train_ds = split['train']eval_ds = split['test']VALID_ACTIONS = [    'do_nothing', 'patch_hr_systems', 'patch_web_server', 'patch_auth_server',    'rotate_credentials', 'isolate_suspicious_host', 'increase_monitoring', 'restore_backup',    'deploy_honeypot', 'phishing_training', 'investigate_top_alert', 'segment_finance_database']ACTION_SET = ', '.join(VALID_ACTIONS)def to_text(example):    prompt = (        'You are CySent BLUE defender. Choose exactly one valid action label from this set: ' + ACTION_SET + '.\n'        + 'Instruction: ' + example['instruction'] + '\n'        + 'State: ' + example['input'] + '\nAction:'    )    return {'text': prompt + ' ' + example['output']}train_text = train_ds.map(to_text, remove_columns=train_ds.column_names)eval_text = eval_ds.map(to_text, remove_columns=eval_ds.column_names)model_name = 'Qwen/Qwen2.5-3B-Instruct'max_seq_length = 512model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_name, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True)model = FastLanguageModel.get_peft_model(model, r=16, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=16, lora_dropout=0.0, bias='none', use_gradient_checkpointing='unsloth', random_state=42)trainer = SFTTrainer(    model=model, tokenizer=tokenizer, train_dataset=train_text, eval_dataset=eval_text, dataset_text_field='text', max_seq_length=max_seq_length,    args=TrainingArguments(output_dir='outputs/cysent_unsloth', per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4, warmup_steps=10, max_steps=200, logging_steps=10, save_steps=100, eval_steps=50, fp16=True, report_to='none'))trainer.train()save_dir = 'outputs/cysent_unsloth_adapter'trainer.model.save_pretrained(save_dir)tokenizer.save_pretrained(save_dir)print('Saved adapter:', save_dir)

In [ ]:
def parse_action(text):    lower = text.lower()    if 'isolate_host' in lower:        return 'isolate_suspicious_host'    for action in VALID_ACTIONS:        if action in lower:            return action    return 'do_nothing'test_state = 'risk=0.72, attack=phishing_email, target=auth server, compromised=1, credential_exposure=0.81'prompt = 'You are CySent BLUE defender. Choose exactly one valid action label from this set: ' + ACTION_SET + '.\nState: ' + test_state + '\nAction:'inputs = tokenizer(prompt, return_tensors='pt').to(trainer.model.device)outputs = trainer.model.generate(**inputs, max_new_tokens=16, do_sample=False)decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)print('Raw output:', decoded)print('Predicted action:', parse_action(decoded))